# Erdős–Rényi Graphs and Subgraph Thresholds

### Topics in Applied Math

Inspired by a course taught by Gergely Odor,
Medical University of Vienna.

## How would you model a “random network”?**

Compare with a neighbor.

## Our model: \(G(n,p)\) · 5 minutes

A **simple, undirected graph** has:

- a vertex set;
- edges joining distinct pairs of vertices;
- no self-loops or repeated edges.

For $G(n,p)$:

1. Fix $n$ labeled vertices.
2. Include each of the $\binom n2$ possible edges independently.
3. Each edge is included with the same probability \(p\).

**Predict:** What happens when $p=0$? When $p=1$?

In [1]:
%pip install --upgrade ipympl ipywidgets


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%matplotlib widget

import itertools
import math

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import ipywidgets as W
from IPython.display import display

# Fixed random numbers shared by all demonstrations.
# For each fixed n and p, this generates the G(n,p) distribution.
# Reusing draws across p gives a monotone coupling.
MAX_N = 600
SEED = 2026
rng = np.random.default_rng(SEED)
U = np.ones((MAX_N, MAX_N))
ii, jj = np.triu_indices(MAX_N, 1)
U[ii, jj] = rng.random(len(ii))
U[jj, ii] = U[ii, jj]


def edge_list(n, p):
    a, b = np.triu_indices(n, 1)
    keep = U[a, b] < p
    return list(zip(a[keep].tolist(), b[keep].tolist()))


def base_positions():
    # Golden-angle placement makes growing n preserve old coordinates.
    k = np.arange(MAX_N)
    angle = k * np.pi * (3 - np.sqrt(5))
    radius = np.sqrt(k + 1)
    return np.column_stack((radius * np.cos(angle),
                            radius * np.sin(angle)))


class GraphCanvas:
    """Interactive graph with stable coordinates and draggable nodes."""

    def __init__(self, n=5, circular=False):
        with plt.ioff():
            self.fig, self.ax = plt.subplots(figsize=(7, 4.8))
        self.pos = base_positions()
        if circular:
            angle = np.linspace(0, 2*np.pi, n, endpoint=False)
            self.pos[:n] = np.c_[np.cos(angle), np.sin(angle)]

        self.n = n
        self.edges = []
        self.dragging = None
        self.manually_moved = set()
        self.fig.canvas.mpl_connect("button_press_event", self.press)
        self.fig.canvas.mpl_connect("motion_notify_event", self.move)
        self.fig.canvas.mpl_connect("button_release_event", self.release)

    def press(self, event):
        if event.inaxes != self.ax or event.button != 1:
            return
        pixel_positions = self.ax.transData.transform(self.pos[:self.n])
        distances = np.linalg.norm(
            pixel_positions - np.array([event.x, event.y]), axis=1
        )
        closest = int(np.argmin(distances))
        if distances[closest] < 15:
            self.dragging = closest

    def move(self, event):
        if self.dragging is None or event.inaxes != self.ax:
            return
        self.pos[self.dragging] = [event.xdata, event.ydata]
        self.manually_moved.add(self.dragging)
        self.draw()

    def release(self, event):
        self.dragging = None

    def update(self, n, edges, title="", relax=False):
        self.n, self.edges, self.title = n, list(edges), title
        if relax and n <= 65 and edges:
            g = nx.Graph()
            g.add_nodes_from(range(n))
            g.add_edges_from(edges)
            kwargs = {}
            fixed = sorted(v for v in self.manually_moved if v < n)
            if fixed:
                kwargs["fixed"] = fixed
            target = nx.spring_layout(
                g,
                pos={v: self.pos[v].copy() for v in range(n)},
                iterations=5,
                seed=SEED,
                scale=max(1, np.sqrt(n)),
                **kwargs
            )
            for v in range(n):
                if v not in self.manually_moved:
                    self.pos[v] = 0.9*self.pos[v] + 0.1*target[v]
        self.draw()

    def draw(self):
        self.ax.clear()
        if self.edges:
            segments = [self.pos[[a, b]] for a, b in self.edges]
            self.ax.add_collection(LineCollection(
                segments, colors="#666666", linewidths=0.9, alpha=0.6
            ))
        self.ax.scatter(
            *self.pos[:self.n].T, s=95 if self.n <= 30 else 30,
            facecolors="white", edgecolors="black", zorder=3
        )
        if self.n <= 30:
            for v, xy in enumerate(self.pos[:self.n]):
                self.ax.annotate(
                    str(v+1), xy, ha="center", va="center",
                    fontsize=8, zorder=4
                )
        xy = self.pos[:self.n]
        center = (xy.max(axis=0) + xy.min(axis=0))/2
        span = max(np.ptp(xy[:, 0]), np.ptp(xy[:, 1]), 2)*0.65
        self.ax.set_xlim(center[0]-span, center[0]+span)
        self.ax.set_ylim(center[1]-span, center[1]+span)
        self.ax.set_aspect("equal")
        self.ax.axis("off")
        self.ax.set_title(getattr(self, "title", ""))
        self.fig.canvas.draw_idle()

In [3]:
import time

def gentle_update(self, n, edges, title="", relax=False):
    self.n = n
    self.edges = list(edges)
    self.title = title

    # Capture initial positions once. These provide a stable spatial scaffold.
    if not hasattr(self, "_anchors"):
        self._anchors = self.pos.copy()

    start = self.pos[:n].copy()
    target = start.copy()

    fixed = np.array([
        v in self.manually_moved for v in range(n)
    ], dtype=bool)

    if relax and n <= 65:
        anchors = self._anchors[:n]
        center = anchors.mean(axis=0)
        radius = max(
            np.linalg.norm(anchors - center, axis=1).max(),
            1.0
        )

        # Preferred local separation. Repulsion prevents dense-graph collapse.
        spacing = radius / np.sqrt(n)

        # Normalize edge forces by degree so adding many edges doesn't
        # suddenly multiply the force acting on a vertex.
        degree = np.zeros(n)
        for a, b in self.edges:
            degree[a] += 1
            degree[b] += 1

        for _ in range(6):
            delta = target[:, None, :] - target[None, :, :]
            distance = np.linalg.norm(delta, axis=2)

            # Guard against coincident nodes; exclude self-interaction.
            distance = np.maximum(distance, 0.05 * spacing)
            np.fill_diagonal(distance, np.inf)

            repulsion = (
                spacing**2
                * delta
                / distance[:, :, None]**2
            ).sum(axis=1)

            attraction = np.zeros_like(target)

            for a, b in self.edges:
                d = target[b] - target[a]
                length = max(np.linalg.norm(d), 1e-9)

                # Springs have a nonzero preferred length.
                weight = 1 / max(degree[a], degree[b], 1)
                force = weight * (length - spacing) * d / length

                attraction[a] += force
                attraction[b] -= force

            # Anchoring keeps the overall drawing recognizable.
            anchoring = 0.8 * (anchors - target)

            displacement = 0.035 * (
                0.35 * repulsion + attraction + anchoring
            )

            # Bound each internal step, regardless of graph density.
            lengths = np.linalg.norm(displacement, axis=1)
            cap = 0.008 * radius
            displacement *= np.minimum(
                1.0, cap / np.maximum(lengths, 1e-12)
            )[:, None]

            displacement[fixed] = 0
            target += displacement

        # Bound total movement per slider callback as well.
        movement = target - start
        lengths = np.linalg.norm(movement, axis=1)
        cap = 0.025 * radius
        movement *= np.minimum(
            1.0, cap / np.maximum(lengths, 1e-12)
        )[:, None]
        target = start + movement

        # One bounded layout update; no sleeping or event-loop manipulation.
        self.pos[:n] = target
        self.draw()

    else:
        self.draw()


GraphCanvas.update = gentle_update

In [4]:
def simulate_edge_decisions():
    pairs5 = list(itertools.combinations(range(5), 2))
    angle5 = np.linspace(0, 2*np.pi, 5, endpoint=False)
    pos5 = np.c_[np.cos(angle5), np.sin(angle5)]

    p_step = W.FloatSlider(
        value=0.2, min=0, max=1, step=0.01, description="p:",
        continuous_update=True
    )
    next_button = W.Button(description="Test next edge")
    reset_button = W.Button(description="Reset decisions")
    step_status = W.HTML()
    state = {"k": 0}

    with plt.ioff():
        step_fig, (graph_ax, ruler_ax) = plt.subplots(
            1, 2, figsize=(9, 4), gridspec_kw={"width_ratios": [3, 2]}
        )


    def draw_steps(*_):
        graph_ax.clear()
        ruler_ax.clear()
        k, p = state["k"], p_step.value

        for index, (a, b) in enumerate(pairs5):
            xy = pos5[[a, b]]
            if index < k:
                if U[a, b] < p:
                    graph_ax.plot(*xy.T, color="black", linewidth=2)
            else:
                graph_ax.plot(*xy.T, "--", color="0.7", linewidth=1)
            if index == k:
                graph_ax.plot(*xy.T, color="#0072B2", linewidth=4, alpha=0.7)

        graph_ax.scatter(*pos5.T, s=450, c="white",
                        edgecolors="black", zorder=3)
        for v, xy in enumerate(pos5):
            graph_ax.text(*xy, str(v+1), ha="center", va="center", zorder=4)
        graph_ax.set_aspect("equal")
        graph_ax.axis("off")
        graph_ax.set_title("Five labeled vertices")

        ruler_ax.plot([0, 1], [0, 0], color="black")
        ruler_ax.plot([0, p], [0, 0], color="#0072B2", linewidth=7)
        ruler_ax.axvline(p, color="#0072B2", linestyle="--")
        ruler_ax.text(p, 0.17, f"p = {p:.2f}", ha="center")
        ruler_ax.set_xlim(-0.05, 1.05)
        ruler_ax.set_ylim(-0.5, 0.5)
        ruler_ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1])
        ruler_ax.set_yticks([])
        ruler_ax.set_title("Reveal the draw by clicking")

        if k:
            a, b = pairs5[k-1]
            u = U[a, b]
            ruler_ax.plot(u, 0, "o", color="black", markersize=9)
            outcome = "INCLUDED" if u < p else "EXCLUDED"
            ruler_ax.text(u, -0.2, f"U = {u:.3f}", ha="center")
            step_status.value = (
                f"Decision {k}/10: edge ({a+1}, {b+1}); "
                f"U = {u:.3f}, p = {p:.2f}: <b>{outcome}</b>."
            )
        else:
            step_status.value = "No draws revealed yet."

        next_button.disabled = k == len(pairs5)
        step_fig.canvas.draw_idle()


    def advance(_):
        state["k"] += 1
        draw_steps()


    def reset_steps(_=None):
        state["k"] = 0
        draw_steps()


    next_button.on_click(advance)
    reset_button.on_click(reset_steps)
    # Changing p restarts the decisions, without changing the random draws.
    p_step.observe(reset_steps, names="value")
    draw_steps()
    display(W.VBox([
        W.HBox([p_step, next_button, reset_button]),
        step_status, step_fig.canvas
    ]))

In [5]:
def manipulate_n_and_p_Gnp_graph():
    live_n = W.IntSlider(
        value=5, min=5, max=100, description="n:",
        continuous_update=True
    )
    live_p = W.FloatSlider(
        value=0.2, min=0, max=1, step=0.01, description="p:",
        continuous_update=True
    )
    relax = W.Checkbox(value=True, description="Gentle layout adjustment")
    live_status = W.HTML()
    live_graph = GraphCanvas(5, circular=True)


    def update_live(*_):
        n, p = live_n.value, live_p.value
        edges = edge_list(n, p)
        live_graph.update(n, edges, f"G({n}, {p:.3f})", relax.value)
        live_status.value = (
            f"Possible edges: {math.comb(n, 2)}; "
            f"realized edges: {len(edges)}; "
            f"expected edges: {math.comb(n, 2)*p:.2f}."
        )


    live_n.observe(update_live, names="value")
    live_p.observe(update_live, names="value")
    relax.observe(update_live, names="value")
    update_live()
    display(W.VBox([
        W.HBox([live_n, live_p]), relax, live_status, live_graph.fig.canvas
    ]))

In [6]:
def dropdown_menu_investigation():
    n0 = 20
    scale_mode = W.Dropdown(
        options=[
            ("Fixed p", "fixed"),
            ("Fixed expected edge count", "constant"),
            ("Expected edges proportional to n", "linear"),
        ],
        value="fixed", description="Rule:",
        style={"description_width": "initial"},
        layout=W.Layout(width="420px")
    )
    scale_n = W.IntSlider(
        value=20, min=5, max=120, description="n:",
        continuous_update=True
    )
    q_slider = W.FloatSlider(
        value=0.1, min=0, max=1, step=0.01,
        description="q at n=20:",
        style={"description_width": "initial"},
        continuous_update=True
    )
    scale_status = W.HTML()
    scale_graph = GraphCanvas(20)


    def update_scale(*_):
        n, q = scale_n.value, q_slider.value
        reference_edges = math.comb(n0, 2)*q
        if scale_mode.value == "fixed":
            raw_p = q
        elif scale_mode.value == "constant":
            raw_p = reference_edges / math.comb(n, 2)
        else:
            raw_p = reference_edges*(n/n0) / math.comb(n, 2)

        p = min(1.0, raw_p)
        edges = edge_list(n, p)
        scale_graph.update(n, edges, f"G({n}, {p:.4f})")
        warning = (
            " <b>Target is infeasible: p has been capped at 1.</b>"
            if raw_p > 1 else ""
        )
        scale_status.value = (
            f"p = {p:.4f}; realized edges = {len(edges)}; "
            f"expected edges = {math.comb(n, 2)*p:.2f}; "
            f"expected average degree = {(n-1)*p:.2f}."
            + warning
        )


    for control in [scale_mode, scale_n, q_slider]:
        control.observe(update_scale, names="value")
    update_scale()
    display(W.VBox([
        scale_mode, W.HBox([scale_n, q_slider]),
        scale_status, scale_graph.fig.canvas
    ]))

In [7]:
def make_t_vs_n_slider():
    exponent = W.FloatSlider(
        value=1.0, min=0, max=3, step=0.02,
        description="t:", continuous_update=True
    )
    power_n = W.IntSlider(
        value=40, min=5, max=MAX_N, description="n:",
        continuous_update=True
    )
    power_status = W.HTML()
    power_graph = GraphCanvas(40)


    def update_power(*_):
        n, t = power_n.value, exponent.value
        p = n**(-t)
        edges = edge_list(n, p)
        power_graph.update(n, edges, f"p = n^(-t) = {p:.4g}")
        power_status.value = (
            f"n = {n}; t = {t:.2f}; p = {p:.5g}; "
            f"realized edges = {len(edges)}; "
            f"expected edges = {math.comb(n, 2)*p:.3g}."
        )


    exponent.observe(update_power, names="value")
    power_n.observe(update_power, names="value")
    update_power()
    display(W.VBox([
        W.HBox([exponent, power_n]), power_status, power_graph.fig.canvas
    ]))

In [8]:
def make_asymptotic_expected_subgraph_copies_chart():
    graph_choice = W.Dropdown(
        options=["Edge", "Path P4", "Triangle K3", "Clique K4"],
        value="Clique K4", description="Subgraph:"
    )
    count_t = W.FloatSlider(
        value=2/3, min=0, max=3, step=0.01,
        description="t:", continuous_update=True
    )
    count_log_n = W.FloatSlider(
        value=3, min=1, max=6, step=0.1,
        description="log10(n):", continuous_update=True
    )
    count_status = W.HTML()

    with plt.ioff():
        count_fig, count_ax = plt.subplots(figsize=(7, 4))


    def parameters():
        return {
            "Edge": (2, 1, 2),
            "Path P4": (4, 3, 2),
            "Triangle K3": (3, 3, 6),
            "Clique K4": (4, 6, 24),
        }[graph_choice.value]


    def log_expected(ns, t, v, e, aut):
        ns = np.asarray(ns, dtype=float)
        log_falling = sum(np.log(ns-k) for k in range(v))
        return (log_falling - math.log(aut) - e*t*np.log(ns))/math.log(10)


    def update_counts(*_):
        v, e, aut = parameters()
        t = count_t.value
        n = max(v, round(10**count_log_n.value))
        ns = np.geomspace(10, 1e6, 250)
        ys = log_expected(ns, t, v, e, aut)
        y = float(log_expected(n, t, v, e, aut))

        count_ax.clear()
        count_ax.plot(np.log10(ns), ys, color="black")
        count_ax.axhline(0, color="0.5", linestyle="--", label="Expected count = 1")
        count_ax.scatter([math.log10(n)], [y], s=60, color="#0072B2")
        count_ax.set_xlabel("log10(n)")
        count_ax.set_ylabel("log10(expected number of copies)")
        count_ax.set_title(graph_choice.value)
        count_ax.grid(alpha=0.2)
        count_ax.legend()
        count_fig.canvas.draw_idle()
        count_status.value = (
            f"n = {n:,}; t = {t:.3f}; critical exponent = {v/e:.3f}; "
            f"power of n = v − te = {v-t*e:.3f}; "
            f"expected copies = {10**y:.4g}."
        )


    for control in [graph_choice, count_t, count_log_n]:
        control.observe(update_counts, names="value")
    update_counts()
    display(W.VBox([
        graph_choice, W.HBox([count_t, count_log_n]),
        count_status, count_fig.canvas
    ]))

In [9]:
simulate_edge_decisions()

## Experiment 2: vary \(n\) and \(p\)

In [10]:
manipulate_n_and_p_Gnp_graph()

In [11]:
dropdown_menu_investigation()

## What if I wanted $\mathbb E[E]=C$?

## What if I wanted $\mathbb E[E]=C$?

Let $I_{ij}=1$ if edge $ij$ is present, and $0$ otherwise.

$$
E=\sum_{i<j}I_{ij},
\qquad
\mathbb E[E]=\sum_{i<j}\mathbb E[I_{ij}]
=\binom n2p.
$$

$$
\implies p=\frac{C}{\binom n2}\sim 2C n^{-2}.
$$

So we have $p$ scaling as $p\propto n^{-2}$.

## Pair discussion: Choose $p$ so that $\mathbb E[E]=an$.

## Pair discussion: Choose $p$ so that $\mathbb E[E]=an$.

$$
p=\frac{an}{\binom n2}
=\frac{2a}{n-1}
\sim 2a n^{-1}.
$$

Another nice property: the expected average degree (edges per node) is

$$
\frac{2\mathbb E[E]}{n}=2a.
$$

## The exponent $t$

Write

$$
p=n^{-t},
\qquad
t=-\frac{\log p}{\log n}.
$$

- $t=0: p=1.$
- $t=1$: expected edges are of order $n$.
- $t=2$: expected edges is asymptotically constant (not dependent on $n$).

**Larger $t$ means smaller $p$. We call the graph more sparse when it has fewer edges.**

In [12]:
make_t_vs_n_slider()

## What does “appearance threshold” mean? · 5 minutes

Fix a graph $H$. We ask whether $G(n,p)$ contains a copy of $H$.

We mean an **ordinary, not necessarily induced, subgraph**:
extra edges among the selected vertices are allowed.

A threshold scale $p_H(n)$ satisfies:

$$
p/p_H\to 0
\quad\Longrightarrow\quad
\Pr(H\subseteq G(n,p))\to 0,
$$

$$
p/p_H\to\infty
\quad\Longrightarrow\quad
\Pr(H\subseteq G(n,p))\to 1.
$$

At $p$ equal to a constant times the threshold scale,
neither conclusion automatically follows.

All limits are as $n\to\infty$, with $H$ fixed.

## Count a path on \(m\) vertices · 5 minutes

Let \(X\) count path copies. Ordering \(m\) distinct vertices gives a path;
reversing that order gives the same copy.

$$
\mathbb E[X]
=\frac{(n)_m}{2}p^{m-1}
\asymp n^m p^{m-1}.
$$

Here $(n)_m=n(n-1)\cdots(n-m+1)$.

Substitute $p=n^{-t}$:

$$
\mathbb E[X]\asymp n^{\,m-t(m-1)}.
$$

The critical exponent is

$$
t_*=\frac{m}{m-1},
\qquad
p_*(n)=n^{-m/(m-1)}.
$$

**Predict:** On which side of $t_*$ should paths disappear?

## What have we proved?

If $t>m/(m-1)$, then $\mathbb E[X]\to0$.

Markov’s inequality gives

$$
\Pr(X\geq1)\leq\mathbb E[X]\to0.
$$

That proves the **nonappearance side**.

If $t<m/(m-1)$, then $\mathbb E[X]\to\infty$.

That is a useful clue, but it is **not by itself a proof**
that $X\geq1$ with high probability.

For paths, a second-moment argument establishes the appearance side.

## Explore expected subgraph counts · 4 minutes

Choose a graph and vary the exponent.

**Questions:**

- When does the expected count grow with \(n\)?
- When does it shrink?
- Why can a finite sample near the threshold be misleading?

This display shows **expectations**, not simulated appearance probabilities.

In [13]:
make_asymptotic_expected_subgraph_copies_chart()

## Structure matters: the densest part controls appearance

For a fixed graph $H$, define

$$
m(H)=\max_{\substack{F\subseteq H\\e(F)>0}}
\frac{e(F)}{v(F)}.
$$

The general threshold scale is

$$
p_H(n)=n^{-1/m(H)}.
$$

**Discussion:** What assumption was hidden in our simpler counting rule?

## Exit ticket · 3 minutes

1. State the three defining assumptions of \(G(n,p)\).
2. Choose \(p(n)\) to keep expected average degree approximately 4.
3. For \(K_4\), is \(t=0.8\) on the appearance or nonappearance side?
4. Explain why a large expected subgraph count is not a complete proof
   of appearance.